#  ⭐ `dim_notificacoes`

In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

In [2]:
project_root

WindowsPath('C:/Users/janet/Documents/VigiMed/vigimed')

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Notificacoes/Notificacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,NOTIFICADOR,ATUALIZADO
0,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300212656,20230928,20230928,None,Notificaçăo espontânea,None,19900131,30 ano,None,None,Feminino,Năo,Năo,68.0,165,Hemiparesia,Sim,Outro efeito clinicamente significativo,Recuperado,20210125,20210206,12 dia,Concomitante,Tamisa,Năo aplicável,Médico,2026-01-15
1,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300208322,20230901,20230901,None,Notificaçăo espontânea,None,None,None,None,None,Feminino,Năo,Năo,None,None,Cefaleia,Năo,Outro efeito clinicamente significativo,Desconhecido,20210122,None,None,Suspeito,CoronaVac,Năo aplicável,Consumidor ou outro năo profissional de saúde,2026-01-15


# 🥈 Silver

normalized data, medium quality

In [5]:
silver = bronze.copy()
silver.shape

(311134, 30)

## ✅ hash

In [6]:
from utils import build_row_hash

silver["HASH_BRONZE"] = build_row_hash(
    silver, 
    silver.columns.difference(['ATUALIZADO']).tolist(), 
    algo="sha256", 
    sep="|"
)

## ✅ Missing

In [7]:
silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [8]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [9]:
hist_silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [10]:
hist_silver.shape

(311134, 31)

## ✅ DATAS

In [11]:
colunas_data = ["DATA_INCLUSAO_SISTEMA", "DATA_ULTIMA_ATUALIZACAO", "DATA_NOTIFICACAO", "DATA_NASCIMENTO", "DATA_INICIO_HORA", "DATA_FINAL_HORA"]

In [12]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  object
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  object
 2   DATA_NOTIFICACAO         159253 non-null  object
 3   DATA_NASCIMENTO          230424 non-null  object
 4   DATA_INICIO_HORA         225681 non-null  object
 5   DATA_FINAL_HORA          157812 non-null  object
dtypes: object(6)
memory usage: 14.2+ MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,20230928,20230928,None,19900131,20210125,20210206
1,20230901,20230901,None,None,20210122,None
2,20231006,20231006,None,19710522,20210203,None
3,20250721,20250721,None,19741123,20210302,None
4,20230927,20230927,None,None,None,None


In [13]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [14]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  datetime64[ns]
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  datetime64[ns]
 2   DATA_NOTIFICACAO         158960 non-null  datetime64[ns]
 3   DATA_NASCIMENTO          229298 non-null  datetime64[ns]
 4   DATA_INICIO_HORA         169043 non-null  datetime64[ns]
 5   DATA_FINAL_HORA          107847 non-null  datetime64[ns]
dtypes: datetime64[ns](6)
memory usage: 14.2 MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,2023-09-28,2023-09-28,NaT,1990-01-31,2021-01-25,2021-02-06
1,2023-09-01,2023-09-01,NaT,NaT,2021-01-22,NaT
2,2023-10-06,2023-10-06,NaT,1971-05-22,2021-02-03,NaT
3,2025-07-21,2025-07-21,NaT,1974-11-23,2021-03-02,NaT
4,2023-09-27,2023-09-27,NaT,NaT,NaT,NaT


In [15]:
hist_silver.shape

(311134, 31)

## ✅ Mappings

- UF
- TIPO_ENTRADA_VIGIMED
- RECEBIDO_DE
- TIPO_NOTIFICACAO
- NOTIFICACAO_PARENT_CHILD
- GRUPO_IDADE
- SEXO
- GESTANTE
- LACTANTE
- GRAVE
- GRAVIDADE
- DESFECHO
- RELACAO_MEDICAMENTO_EVENTO
- ACAO_ADOTADA
- NOTIFICADOR

In [16]:
# UF
uf_map = {
    "AC": 1, "AL": 2, "AP": 3, "AM": 4, "BA": 5, "CE": 6, "DF": 7, "ES": 8, "GO": 9, "MA": 10, 
    "MT": 11, "MS": 12, "MG": 13, "PA": 14, "PB": 15, "PR": 16, "PE": 17, "PI": 18, "RJ": 19,
    "RN": 20, "RS": 21, "RO": 22, "RR": 23, "SC": 24, "SP": 25, "SE": 26, "TO": 27
}

# TIPO_ENTRADA_VIGIMED
tipo_entrada_vigimed_map = {
    "SERVICOS DE SAUDE": 1, "EMPRESAS FARMACEUTICAS": 2, "PACIENTES E PROFISSIONAIS DE SAUDE": 3,
    "SERVICOS DE VACINACAO": 4, "EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS": 5,
    "EREPORTING - INDUSTRIA, CARGA E2B": 6, "VIGIMOBILE": 7, "VIGIFLOW EFORMS": 8
}

# RECEBIDO_DE
recebido_de_map = {
    "CARGA E2B": 1, "ENTRADA MANUAL DE DADOS": 2, "AUTORIDADE REGULADORA": 3,
    "CENTRO REGIONAL DE FARMACOVIGILANCIA": 4, "EMPRESA FARMACEUTICA": 5,
    "OUTRO (P.EX. DISTRIBUIDORA, FINANCIADOR DE ESTUDO, ORGANIZACAO REPRESENTATIVA PARA PESQUISA CLINICA, OU ORGANIZACAO NAO-COMERCIAL)": 6,
    "PACIENTE/CONSUMIDOR": 7, "PROFISSIONAL DE SAUDE": 8
}

# TIPO_NOTIFICACAO
tipo_notificacao_map = {
    "NOTIFICACAO ESPONTANEA": 1, "NOTIFICACAO DE ESTUDO": 2, "OUTRO": 3
}

# NOTIFICACAO_PARENT_CHILD
notificacao_parent_child_map = {
    "SIM": 1, "NAO": 2
}

# GRUPO_IDADE
grupo_idade_map = {
    "FETO": 1, "NEONATO": 2, "INFANTIL": 3, "CRIANCA": 4, "ADOLESCENTE": 5,
    "ADULTO": 6, "IDOSO": 7, "OUTRO": 8
}

# SEXO
sexo_map = {
    "FEMININO": 1, "MASCULINO": 2
}

# GESTANTE
gestante_map = {
    "SIM": 1, "NAO": 2
}

# LACTANTE
lactante_map = {
    "SIM": 1, "NAO": 2
}

# GRAVE
grave_map = {
    "SIM": 1, "NAO": 2
}

# GRAVIDADE
gravidade_map = {
    "OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO": 1, "HOSPITALIZACAO": 2, "AMEACA R VIDA": 3,
    "RESULTOU EM OBITO": 4, "INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA": 5,
    "ANOMALIA CONGENITA OU MALFORMACAO AO NASCER": 6
}

# DESFECHO
desfecho_map = {
    "RECUPERADO": 1, "EM RECUPERACAO": 2, "NAO RECUPERADO": 3, "FATAL": 4
}

# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1, "CONCOMITANTE": 2, "INTERACAO": 3, "MEDICAMENTO NAO ADMINISTRADO": 4
}

# ACAO_ADOTADA
acao_adotada_map = {
    "AUMENTO DA DOSE": 1, "NAO APLICAVEL": 2, "REDUCAO DA DOSE": 3, 
    "RETIRADA DO MEDICAMENTO": 4, "SEM ALTERACAO DA DOSE": 5
}

# NOTIFICADOR
notificador_map = {
    "ADVOGADO": 1, "CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE": 2,
    "FARMACEUTICO": 3, "MEDICO": 4, "OUTRO PROFISSIONAL DE SAUDE": 5
}


mappings_config = [
    {
        "kind": "dict",
        "col": "UF",
        "map": uf_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "TIPO_ENTRADA_VIGIMED",
        "map": tipo_entrada_vigimed_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "RECEBIDO_DE",
        "map": recebido_de_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "TIPO_NOTIFICACAO",
        "map": tipo_notificacao_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "NOTIFICACAO_PARENT_CHILD",
        "map": notificacao_parent_child_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRUPO_IDADE",
        "map": grupo_idade_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "SEXO",
        "map": sexo_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GESTANTE",
        "map": gestante_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "LACTANTE",
        "map": lactante_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRAVE",
        "map": grave_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "GRAVIDADE",
        "map": gravidade_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "DESFECHO",
        "map": desfecho_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "RELACAO_MEDICAMENTO_EVENTO",
        "map": relacao_medicamento_evento_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "ACAO_ADOTADA",
        "map": acao_adotada_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
    {
        "kind": "dict",
        "col": "NOTIFICADOR",
        "map": notificador_map,
        "fillna_valor": "DESCONHECIDO",
        "fillna_chave": 0,
        "tipo_int64": True,
        "drop_original": False,
    },
]

In [17]:
hist_silver = apply_mappings(hist_silver, mappings_config)

In [18]:
hist_silver.shape

(311134, 61)

In [19]:
hist_silver[['UF','UF_VALOR','UF_CHAVE']].value_counts(dropna=False)

UF   UF_VALOR      UF_CHAVE
SP   SP            25          117317
NaN  DESCONHECIDO  0            81507
MG   MG            13           20254
BA   BA            5            11452
RJ   RJ            19           10871
DF   DF            7            10717
SC   SC            24            9291
RS   RS            21            8592
CE   CE            6             7915
PR   PR            16            7564
GO   GO            9             5921
ES   ES            8             5518
RN   RN            20            3691
PI   PI            18            1697
MA   MA            10            1623
PA   PA            14            1439
PE   PE            17            1398
AM   AM            4             1366
PB   PB            15             870
MS   MS            12             581
MT   MT            11             398
SE   SE            26             396
AL   AL            2              316
 C    C            0              248
RO   RO            22             101
TO   TO            27 

In [20]:
hist_silver[['TIPO_ENTRADA_VIGIMED', 'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE']].value_counts(dropna=False)

TIPO_ENTRADA_VIGIMED                             TIPO_ENTRADA_VIGIMED_VALOR                       TIPO_ENTRADA_VIGIMED_CHAVE
Serviços de Saúde                                SERVIÇOS DE SAÚDE                                1                             152367
Empresas Farmacęuticas                           EMPRESAS FARMACĘUTICAS                           2                              79444
Pacientes e Profissionais de Saúde               PACIENTES E PROFISSIONAIS DE SAÚDE               3                              37774
eReporting - Indústria, Entrada manual de dados  EREPORTING - INDÚSTRIA, ENTRADA MANUAL DE DADOS  5                              11710
eReporting - Indústria, Carga E2B                EREPORTING - INDÚSTRIA, CARGA E2B                6                               9996
Serviços de Vacinaçăo                            SERVIÇOS DE VACINAÇĂO                            4                               9321
VigiMobile                                       VIGIMOBILE      

In [21]:
hist_silver[['RECEBIDO_DE', 'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE']].value_counts(dropna=False)

RECEBIDO_DE                                                                                                                         RECEBIDO_DE_VALOR                                                                                                                   RECEBIDO_DE_CHAVE
Profissional de Saúde                                                                                                               PROFISSIONAL DE SAÚDE                                                                                                               8                    104704
Empresa Farmacęutica                                                                                                                EMPRESA FARMACĘUTICA                                                                                                                5                    101702
NaN                                                                                                                                 DE

In [22]:
hist_silver[['TIPO_NOTIFICACAO', 'TIPO_NOTIFICACAO_VALOR', 'TIPO_NOTIFICACAO_CHAVE']].value_counts(dropna=False)

TIPO_NOTIFICACAO                                TIPO_NOTIFICACAO_VALOR                          TIPO_NOTIFICACAO_CHAVE
Notificaçăo espontânea                          NOTIFICAÇĂO ESPONTÂNEA                          1                         268975
Notificaçăo de estudo                           NOTIFICAÇĂO DE ESTUDO                           2                          22119
Outro                                           OUTRO                                           3                          19855
Năo disponível pelo notificador (desconhecido)  NĂO DISPONÍVEL PELO NOTIFICADOR (DESCONHECIDO)  0                            185
Name: count, dtype: int64

In [23]:
hist_silver[['NOTIFICACAO_PARENT_CHILD', 'NOTIFICACAO_PARENT_CHILD_VALOR', 'NOTIFICACAO_PARENT_CHILD_CHAVE']].value_counts(dropna=False)

NOTIFICACAO_PARENT_CHILD  NOTIFICACAO_PARENT_CHILD_VALOR  NOTIFICACAO_PARENT_CHILD_CHAVE
NaN                       DESCONHECIDO                    0                                 310006
Sim                       SIM                             1                                   1128
Name: count, dtype: int64

In [24]:
hist_silver[['GRUPO_IDADE', 'GRUPO_IDADE_VALOR', 'GRUPO_IDADE_CHAVE']].value_counts(dropna=False)

GRUPO_IDADE  GRUPO_IDADE_VALOR  GRUPO_IDADE_CHAVE
NaN          DESCONHECIDO       0                    183974
Adulto       ADULTO             6                     71598
Idoso        IDOSO              7                     42827
Criança      CRIANÇA            4                      4389
Infantil     INFANTIL           3                      3734
Adolescente  ADOLESCENTE        5                      3139
Neonato      NEONATO            2                      1409
Feto         FETO               1                        64
Name: count, dtype: int64

In [25]:
hist_silver[['SEXO', 'SEXO_VALOR', 'SEXO_CHAVE']].value_counts(dropna=False)

SEXO          SEXO_VALOR    SEXO_CHAVE
Feminino      FEMININO      1             186527
Masculino     MASCULINO     2             111891
NaN           DESCONHECIDO  0              11687
Desconhecido  DESCONHECIDO  0               1029
Name: count, dtype: int64

In [26]:
hist_silver[['GESTANTE', 'GESTANTE_VALOR', 'GESTANTE_CHAVE']].value_counts(dropna=False)

GESTANTE  GESTANTE_VALOR  GESTANTE_CHAVE
NaN       DESCONHECIDO    0                 160811
Năo       NĂO             2                 148633
Sim       SIM             1                   1690
Name: count, dtype: int64

In [27]:
hist_silver[['LACTANTE', 'LACTANTE_VALOR', 'LACTANTE_CHAVE']].value_counts(dropna=False)

LACTANTE  LACTANTE_VALOR  LACTANTE_CHAVE
NaN       DESCONHECIDO    0                 160814
Năo       NĂO             2                 148497
Sim       SIM             1                   1823
Name: count, dtype: int64

In [28]:
hist_silver[['GRAVE', 'GRAVE_VALOR', 'GRAVE_CHAVE']].value_counts(dropna=False)

GRAVE  GRAVE_VALOR   GRAVE_CHAVE
Sim    SIM           1              136768
Năo    NĂO           2              105074
NaN    DESCONHECIDO  0               69292
Name: count, dtype: int64

In [29]:
hist_silver[['GRAVIDADE', 'GRAVIDADE_VALOR', 'GRAVIDADE_CHAVE']].value_counts(dropna=False)

GRAVIDADE                                    GRAVIDADE_VALOR                              GRAVIDADE_CHAVE
NaN                                          DESCONHECIDO                                 0                  168932
Outro efeito clinicamente significativo      OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO      1                   82979
Hospitalizaçăo                               HOSPITALIZAÇĂO                               2                   31898
Resultou em óbito                            RESULTOU EM ÓBITO                            4                   11049
Ameaça ŕ vida                                AMEAÇA Ŕ VIDA                                3                    9952
Incapacidade persistente ou significativa    INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA    5                    6179
Anomalia congęnita ou malformaçăo ao nascer  ANOMALIA CONGĘNITA OU MALFORMAÇĂO AO NASCER  6                     145
Name: count, dtype: int64

In [30]:
hist_silver[['DESFECHO', 'DESFECHO_VALOR', 'DESFECHO_CHAVE']].value_counts(dropna=False)

DESFECHO        DESFECHO_VALOR  DESFECHO_CHAVE
Recuperado      RECUPERADO      1                 146660
Desconhecido    DESCONHECIDO    0                  53293
NaN             DESCONHECIDO    0                  49933
Em recuperaçăo  EM RECUPERAÇĂO  2                  28389
Năo Recuperado  NĂO RECUPERADO  3                  21947
Fatal           FATAL           4                  10912
Name: count, dtype: int64

In [31]:
hist_silver[['RELACAO_MEDICAMENTO_EVENTO', 'RELACAO_MEDICAMENTO_EVENTO_VALOR', 'RELACAO_MEDICAMENTO_EVENTO_CHAVE']].value_counts(dropna=False)

RELACAO_MEDICAMENTO_EVENTO    RELACAO_MEDICAMENTO_EVENTO_VALOR  RELACAO_MEDICAMENTO_EVENTO_CHAVE
Suspeito                      SUSPEITO                          1                                   300007
Medicamento năo administrado  MEDICAMENTO NĂO ADMINISTRADO      4                                     8418
Concomitante                  CONCOMITANTE                      2                                     1944
Interaçăo                     INTERAÇĂO                         3                                      764
NaN                           DESCONHECIDO                      0                                        1
Name: count, dtype: int64

In [32]:
hist_silver[['ACAO_ADOTADA', 'ACAO_ADOTADA_VALOR', 'ACAO_ADOTADA_CHAVE']].value_counts(dropna=False)

ACAO_ADOTADA             ACAO_ADOTADA_VALOR       ACAO_ADOTADA_CHAVE
NaN                      DESCONHECIDO             0                     126091
Retirada do medicamento  RETIRADA DO MEDICAMENTO  4                      71728
Năo aplicável            NĂO APLICÁVEL            2                      37552
Desconhecido             DESCONHECIDO             0                      35680
Sem alteraçăo da dose    SEM ALTERAÇĂO DA DOSE    5                      31763
Reduçăo da dose          REDUÇĂO DA DOSE          3                       5609
Aumento da dose          AUMENTO DA DOSE          1                       2711
Name: count, dtype: int64

In [33]:
hist_silver[['NOTIFICADOR', 'NOTIFICADOR_VALOR', 'NOTIFICADOR_CHAVE']].value_counts(dropna=False)

NOTIFICADOR                                    NOTIFICADOR_VALOR                              NOTIFICADOR_CHAVE
Farmacęutico                                   FARMACĘUTICO                                   3                    121082
Consumidor ou outro năo profissional de saúde  CONSUMIDOR OU OUTRO NĂO PROFISSIONAL DE SAÚDE  2                     85054
Outro profissional de saúde                    OUTRO PROFISSIONAL DE SAÚDE                    5                     65862
Médico                                         MÉDICO                                         4                     20474
NaN                                            DESCONHECIDO                                   0                     18164
Advogado                                       ADVOGADO                                       1                       498
Name: count, dtype: int64

## ✅ IDADE_MOMENTO_REACAO

In [34]:
from utils.not_normalize_idade_momento_reacao import normalize_idade_momento_reacao

In [35]:
hist_silver = normalize_idade_momento_reacao(hist_silver, coluna="IDADE_MOMENTO_REACAO")

In [36]:
hist_silver[['IDADE_MOMENTO_REACAO','IDADE_MOMENTO_REACAO_TIPO_CHAVE','IDADE_MOMENTO_REACAO_TIPO_VALOR','IDADE_MOMENTO_REACAO_VALOR']].value_counts(dropna=False).head(10)

IDADE_MOMENTO_REACAO  IDADE_MOMENTO_REACAO_TIPO_CHAVE  IDADE_MOMENTO_REACAO_TIPO_VALOR  IDADE_MOMENTO_REACAO_VALOR
NaN                   0                                DESCONHECIDO                     NaN                           110538
60 ano                1                                ANO                              60.0                            3481
40 ano                1                                ANO                              40.0                            3312
42 ano                1                                ANO                              42.0                            3281
65 ano                1                                ANO                              65.0                            3279
61 ano                1                                ANO                              61.0                            3276
59 ano                1                                ANO                              59.0                            3275
63 ano    

In [37]:
hist_silver.shape

(311134, 64)

## ✅ DURACAO

In [38]:
from utils.not_normalize_duracao import normalize_duracao

In [39]:
hist_silver = normalize_duracao(hist_silver, coluna="DURACAO")

In [40]:
hist_silver[['DURACAO','DURACAO_TIPO_CHAVE','DURACAO_TIPO_VALOR','DURACAO_VALOR']].value_counts(dropna=False).head(10)

DURACAO    DURACAO_TIPO_CHAVE  DURACAO_TIPO_VALOR  DURACAO_VALOR
NaN        0                   DESCONHECIDO        NaN              204258
           0                   DESCONHECIDO        NaN               31565
1 dia      4                   DIA                 1.0               16744
2 dia      4                   DIA                 2.0                5763
1 hora     3                   HORA                1.0                4157
30 minuto  2                   MINUTO              30.0               3884
3 dia      4                   DIA                 3.0                3616
4 dia      4                   DIA                 4.0                2357
5 dia      4                   DIA                 5.0                2002
2 hora     3                   HORA                2.0                1870
Name: count, dtype: int64

In [41]:
hist_silver.shape

(311134, 67)

## ✅ IDADE_GESTACIONAL_MOMENTO_REACAO

In [42]:
from utils.not_normalize_idade_gestacional_momento_reacao import normalize_idade_gestacional_momento_reacao

In [43]:
hist_silver = normalize_idade_gestacional_momento_reacao(hist_silver, coluna="IDADE_GESTACIONAL_MOMENTO_REACAO")

In [44]:
hist_silver[['IDADE_GESTACIONAL_MOMENTO_REACAO','IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE','IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR','IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR']].value_counts(dropna=False).head(10)

IDADE_GESTACIONAL_MOMENTO_REACAO  IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE  IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR  IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR
NaN                               0                                            DESCONHECIDO                                 NaN                                       310802
1 Trimestre                       3                                            TRIMESTRE                                    1.0                                           20
34 Semanas                        1                                            SEMANA                                       34.0                                          19
35 Semanas                        1                                            SEMANA                                       35.0                                          16
9 Semanas                         1                                            SEMANA                                       9.0                  

In [45]:
hist_silver.shape

(311134, 70)

In [46]:
hist_silver.head(20) # verificar caracteres especiais

,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,NOTIFICADOR,ATUALIZADO,HASH_BRONZE,UF_VALOR,UF_CHAVE,TIPO_ENTRADA_VIGIMED_VALOR,TIPO_ENTRADA_VIGIMED_CHAVE,RECEBIDO_DE_VALOR,RECEBIDO_DE_CHAVE,TIPO_NOTIFICACAO_VALOR,TIPO_NOTIFICACAO_CHAVE,NOTIFICACAO_PARENT_CHILD_VALOR,NOTIFICACAO_PARENT_CHILD_CHAVE,GRUPO_IDADE_VALOR,GRUPO_IDADE_CHAVE,SEXO_VALOR,SEXO_CHAVE,GESTANTE_VALOR,GESTANTE_CHAVE,LACTANTE_VALOR,LACTANTE_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,GRAVIDADE_VALOR,GRAVIDADE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,RELACAO_MEDICAMENTO_EVENTO_VALOR,RELACAO_MEDICAMENTO_EVENTO_CHAVE,ACAO_ADOTADA_VALOR,ACAO_ADOTADA_CHAVE,NOTIFICADOR_VALOR,NOTIFICADOR_CHAVE,IDADE_MOMENTO_REACAO_TIPO_CHAVE,IDADE_MOMENTO_REACAO_TIPO_VALOR,IDADE_MOMENTO_REACAO_VALOR,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE,IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR,IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR
0,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300212656,2023-09-28,2023-09-28,NaT,Notificaçăo espontânea,None,1990-01-31,30 ano,None,None,Feminino,Năo,Năo,68.0,165,Hemiparesia,Sim,Outro efeito clinicamente significativo,Recuperado,2021-01-25,2021-02-06,12 dia,Concomitante,Tamisa,Năo aplicável,Médico,2026-01-15,c540f6f2873cded948f9a6156d6a92cf9f59b7e110f84ebc4a988ccdb13b1c3d,SP,25,EMPRESAS FARMACĘUTICAS,2,EMPRESA FARMACĘUTICA,5,NOTIFICAÇĂO ESPONTÂNEA,1,DESCONHECIDO,0,DESCONHECIDO,0,FEMININO,1,NĂO,2,NĂO,2,SIM,1,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,1,RECUPERADO,1,CONCOMITANTE,2,NĂO APLICÁVEL,2,MÉDICO,4,1,ANO,30.0,DIA,4,12.0,0,DESCONHECIDO,NaN
1,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300208322,2023-09-01,2023-09-01,NaT,Notificaçăo espontânea,None,NaT,None,None,None,Feminino,Năo,Năo,None,None,Cefaleia,Năo,Outro efeito clinicamente significativo,Desconhecido,2021-01-22,NaT,None,Suspeito,CoronaVac,Năo aplicável,Consumidor ou outro năo profissional de saúde,2026-01-15,1f6805d588f843fe6f27dc43ae2026701e8d0400f3a10108c262015650b0b5b5,SP,25,EMPRESAS FARMACĘUTICAS,2,EMPRESA FARMACĘUTICA,5,NOTIFICAÇĂO ESPONTÂNEA,1,DESCONHECIDO,0,DESCONHECIDO,0,FEMININO,1,NĂO,2,NĂO,2,NĂO,2,OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO,1,DESCONHECIDO,0,SUSPEITO,1,NĂO APLICÁVEL,2,CONSUMIDOR OU OUTRO NĂO PROFISSIONAL DE SAÚDE,2,0,DESCONHECIDO,NaN,DESCONHECIDO,0,NaN,0,DESCONHECIDO,NaN
2,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300214015,2023-10-06,2023-10-06,NaT,Notificaçăo espontânea,None,1971-05-22,49 ano,None,None,Masculino,Năo,Năo,None,None,Nefrolitíase,Sim,Hospitalizaçăo,Desconhecido,2021-02-03,NaT,None,Concomitante,Metformina,None,Médico,2026-01-15,0bf848a27d155b0a8bdebe8473dfb8c83b42d7d1e9123a666eceea3d811df4f9,SP,25,EMPRESAS FARMACĘUTICAS,2,EMPRESA FARMACĘUTICA,5,NOTIFICAÇĂO ESPONTÂNEA,1,DESCONHECIDO,0,DESCONHECIDO,0,MASCULINO,2,NĂO,2,NĂO,2,SIM,1,HOSPITALIZAÇĂO,2,DESCONHECIDO,0,CONCOMITANTE,2,DESCONHECIDO,0,MÉDICO,4,1,ANO,49.0,DESCONHECIDO,0,NaN,0,DESCONHECIDO,NaN
3,SP,"eReporting - Indústria, Carga E2B",Empresa Farmacęutica,BR-ANVISA-300345102,2025-07-21,2025-07-21,NaT,Notificaçăo espontânea,None,1974-11-23,46 ano,None,None,Masculino,Năo,Năo,None,None,Choque anafilático,Sim,Ameaça ŕ vida,Desconhecido,2021-03-02,NaT,None,Suspeito,Soro antibotropico (pentavalente),Desconhecido,Farmacęutico,2026-01-15,14d6e7f81906f9def902e0ce10e10c8c1c8139bda38a39a700309ff9bd00dc03,SP,25,"EREPORTING - INDÚSTRIA, CARGA E2B",6,EMPRESA FARMACĘUTICA,5,NOTIFICAÇĂO ESPONTÂNEA,1,DESCONHECIDO,0,DESCONHECIDO,0,MASCULINO,2,NĂO,2,NĂO,2,SIM,1,AMEAÇA Ŕ VIDA,3,DESCONHECIDO,0,SUSPEITO,1,DESCONHECIDO,0,FARMACĘUTICO,3,1,ANO,46.0,DESCONHECIDO,0,NaN,0,DESCO

In [47]:
### ✅ CODIGO_ATC

In [48]:
from utils import desagrupar_colunas_pipe

In [49]:
colunas_para_desagrupar = ['ATC_CODE_LEVEL_4']
print(f"Shape antes da desagregação: {hist_silver.shape}") 
hist_silver_dedup = desagrupar_colunas_pipe(hist_silver.rename(columns={'CODIGO_ATC': 'ATC_CODE_LEVEL_4'}), colunas_para_desagrupar)
print(f"Shape depois da desagregação: {hist_silver_dedup.shape}") 

Shape antes da desagregação: (311134, 70)
Shape depois da desagregação: (311134, 70)


In [50]:
hist_silver_dedup["HASH_SILVER"] = build_row_hash(
    hist_silver_dedup, 
    hist_silver_dedup.columns.difference(['ATUALIZADO', 'HASH_BRONZE']).tolist(), 
    algo="sha256", 
    sep="|"
)

In [51]:
hist_silver_dedup.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO', 'HASH_BRONZE', 'UF_VALOR',
       'UF_CHAVE', 'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE',
       'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE', 'TIPO_NOTIFICACAO_VALOR',
       'TIPO_NOTIFICACAO_CHAVE', 'NOTIFICACAO_PARENT_CHILD_VALOR',
       'NOTIFICACAO_PARENT_CHILD_CHAVE', 'GRUPO_IDADE_VALOR',
       'GRUPO_IDADE_CHAVE', 'SEXO_VALOR', 'SEXO_CHAVE', 'GESTANT

In [52]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/02_silver/hist_dim_notificacoes/hist_dim_notificacoes.parquet", index=False)

In [53]:
hist_silver_dedup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 71 columns):
 #   Column                                       Non-Null Count   Dtype         
---  ------                                       --------------   -----         
 0   UF                                           229627 non-null  object        
 1   TIPO_ENTRADA_VIGIMED                         311064 non-null  object        
 2   RECEBIDO_DE                                  255468 non-null  object        
 3   IDENTIFICACAO_NOTIFICACAO                    311134 non-null  object        
 4   DATA_INCLUSAO_SISTEMA                        311134 non-null  datetime64[ns]
 5   DATA_ULTIMA_ATUALIZACAO                      311064 non-null  datetime64[ns]
 6   DATA_NOTIFICACAO                             158960 non-null  datetime64[ns]
 7   TIPO_NOTIFICACAO                             311134 non-null  object        
 8   NOTIFICACAO_PARENT_CHILD                     1128 non-null    ob

# 🥇 Gold

In [54]:
hist_silver_dedup.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO', 'HASH_BRONZE', 'UF_VALOR',
       'UF_CHAVE', 'TIPO_ENTRADA_VIGIMED_VALOR', 'TIPO_ENTRADA_VIGIMED_CHAVE',
       'RECEBIDO_DE_VALOR', 'RECEBIDO_DE_CHAVE', 'TIPO_NOTIFICACAO_VALOR',
       'TIPO_NOTIFICACAO_CHAVE', 'NOTIFICACAO_PARENT_CHILD_VALOR',
       'NOTIFICACAO_PARENT_CHILD_CHAVE', 'GRUPO_IDADE_VALOR',
       'GRUPO_IDADE_CHAVE', 'SEXO_VALOR', 'SEXO_CHAVE', 'GESTANT

In [ ]:
# Lista base de colunas (use sempre os nomes de origem aqui)
gold_cols = [
    'UF',
    'UF_CHAVE',
    'UF_VALOR',

    'TIPO_ENTRADA_VIGIMED',
    'TIPO_ENTRADA_VIGIMED_CHAVE',
    'TIPO_ENTRADA_VIGIMED_VALOR',

    'RECEBIDO_DE',
    'RECEBIDO_DE_CHAVE',
    'RECEBIDO_DE_VALOR',

    'IDENTIFICACAO_NOTIFICACAO',

    'DATA_INCLUSAO_SISTEMA',
    'DATA_ULTIMA_ATUALIZACAO',
    'DATA_NOTIFICACAO',

    'TIPO_NOTIFICACAO',
    'TIPO_NOTIFICACAO_CHAVE',
    'TIPO_NOTIFICACAO_VALOR',

    'NOTIFICACAO_PARENT_CHILD',
    'NOTIFICACAO_PARENT_CHILD_CHAVE',
    'NOTIFICACAO_PARENT_CHILD_VALOR',

    'DATA_NASCIMENTO',

    'IDADE_MOMENTO_REACAO',
    'IDADE_MOMENTO_REACAO_TIPO_CHAVE',
    'IDADE_MOMENTO_REACAO_TIPO_VALOR',
    'IDADE_MOMENTO_REACAO_VALOR',

    'GRUPO_IDADE',
    'GRUPO_IDADE_CHAVE',
    'GRUPO_IDADE_VALOR',

    'IDADE_GESTACIONAL_MOMENTO_REACAO',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_CHAVE',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_TIPO_VALOR',
    'IDADE_GESTACIONAL_MOMENTO_REACAO_VALOR',

    'SEXO',
    'SEXO_CHAVE',
    'SEXO_VALOR',

    'GESTANTE',
    'GESTANTE_CHAVE',
    'GESTANTE_VALOR',

    'LACTANTE',
    'LACTANTE_CHAVE',
    'LACTANTE_VALOR',

    'PESO_KG',
    'ALTURA_CM',

    'REACAO_EVENTO_ADVERSO_MEDDRA',

    'GRAVE',
    'GRAVE_CHAVE',
    'GRAVE_VALOR',

    'GRAVIDADE',
    'GRAVIDADE_CHAVE',
    'GRAVIDADE_VALOR',

    'DESFECHO',
    'DESFECHO_CHAVE',
    'DESFECHO_VALOR',

    'DATA_INICIO_HORA',
    'DATA_FINAL_HORA',

    'DURACAO',
    'DURACAO_TIPO_CHAVE',
    'DURACAO_TIPO_VALOR',
    'DURACAO_VALOR',

    'RELACAO_MEDICAMENTO_EVENTO',
    'RELACAO_MEDICAMENTO_EVENTO_CHAVE',
    'RELACAO_MEDICAMENTO_EVENTO_VALOR',

    'NOME_MEDICAMENTO_WHODRUG',

    'ACAO_ADOTADA',
    'ACAO_ADOTADA_CHAVE',
    'ACAO_ADOTADA_VALOR',

    'NOTIFICADOR',
    'NOTIFICADOR_CHAVE',
    'NOTIFICADOR_VALOR',

    'ATUALIZADO', 
    'HASH_BRONZE',
    'HASH_SILVER'
]

# Dicionário de renomes: chave = nome original, valor = novo nome
rename_map = {
    'LLT_CHAVE_x': 'INDICACAO_MEDDRA_LLT_CHAVE',
    'LLT_CHAVE_y': 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL_LLT_CHAVE',
    # adicione outras renomeações aqui, ex:
    # 'NUMELO_LOTE': 'NUMERO_LOTE',
}

# Lista final de colunas já com renomeação aplicada
final_cols = [rename_map.get(col, col) for col in gold_cols]

# Opcional: se quiser aplicar no DataFrame pandas
fat = hist_silver_dedup.rename(columns=rename_map)
fat = fat[final_cols]

In [60]:
fat["HASH_GOLD"] = build_row_hash(fat, final_cols, algo="sha256", sep="|")

In [61]:
hist_silver_dedup.to_parquet(Path(project_root) / "data/03_gold/dim_notificacoes/dim_notificacoes.parquet", index=False)